# YOLOv8n → INT8 TFLite (320×320) for the Raspberry Pi 5

Run all cells top to bottom (**Runtime → Run all**). CPU runtime is enough.

What it does
1. installs Ultralytics 8.3 (its TFLite export goes PyTorch → ONNX → `onnx2tf` → TFLite and uses Colab's TensorFlow)
2. asks you to upload a zip of calibration frames (VisDrone frames that are **not** in the evaluation sequence, e.g. `uav0000117_02622_v`)
3. exports YOLOv8n with full-integer INT8 quantisation calibrated on those frames
4. **checks** every produced `.tflite` against the float32 TFLite on the same frames with the same decoding code as `edge/local_detector.py`, and refuses models that are broken (the Edge Impulse failure: all class scores or all boxes collapsed)
5. downloads the best INT8 model as `yolov8n_int8_320.tflite`

Why Ultralytics' own export: for TFLite it normalises the boxes *inside* the head before the stride multiplication, so no intermediate tensor reaches 0..320 and boxes and class scores fit the same INT8 scale.

Make the zip on the Mac first:
```
cd TinyML_HT2026/data && zip -qr calib_frames.zip uav0000117_02622_v
```

In [ ]:
# 1. Install (about 2-4 min). Dependencies are installed here with plain pip because
#    Ultralytics' own auto-install uses an NVIDIA package index that Colab cannot reach.
#    If you re-run this notebook: Runtime -> Restart session first, then Run all.
import os
os.environ["YOLO_AUTOINSTALL"] = "False"     # never let Ultralytics try to pip-install mid-export
os.environ["TF_USE_LEGACY_KERAS"] = "1"      # onnx2tf needs Keras 2 (tf_keras)
import tensorflow as tf
TFV = ".".join(tf.__version__.split(".")[:2])
print("Colab TensorFlow", tf.__version__)
!pip install -q "ultralytics>=8.3,<8.4" "onnx2tf>=1.26.3" "sng4onnx>=1.0.1" "onnx_graphsurgeon>=0.3.26" \
    "onnx>=1.12.0,<2.0.0" "onnxslim>=0.1.71" onnxruntime "ai-edge-litert>=1.2.0" "tf_keras~={TFV}.0"
import importlib, sys, ultralytics
for mod in ["onnx2tf", "onnx_graphsurgeon", "sng4onnx", "tf_keras", "onnx", "onnxslim"]:
    m = importlib.import_module(mod)
    print(f"  {mod:18s} {getattr(m, '__version__', 'ok')}")
print("ultralytics", ultralytics.__version__, "| tensorflow", tf.__version__, "| python", sys.version.split()[0])

In [ ]:
# 2. Upload calib_frames.zip (a folder of .jpg frames). If you skip the upload (Cancel),
#    COCO128 is used for calibration instead (works, slightly worse for aerial frames).
import os, glob, zipfile, shutil, random
from google.colab import files

IMGSZ = 320
N_CALIB = 300          # frames used for INT8 calibration (Ultralytics recommends >= 300)
CALIB = "/content/calib"

shutil.rmtree(CALIB, ignore_errors=True)
os.makedirs(f"{CALIB}/images", exist_ok=True)
try:
    up = files.upload()
except Exception as e:
    up = {}
    print("upload skipped:", e)
frames = []
for name in up:
    if name.endswith(".zip"):
        with zipfile.ZipFile(name) as z:
            z.extractall("/content/upload")
frames = sorted(glob.glob("/content/upload/**/*.jpg", recursive=True) + glob.glob("/content/upload/**/*.png", recursive=True))
print(f"{len(frames)} uploaded frames")

if frames:
    random.seed(0)
    pick = frames if len(frames) <= N_CALIB else sorted(random.sample(frames, N_CALIB))
    for f in pick:
        shutil.copy(f, f"{CALIB}/images/{os.path.basename(f)}")
    COCO = ['person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball','kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake','chair','couch','potted plant','bed','dining table','toilet','tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven','toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear','hair drier','toothbrush']
    with open("/content/calib.yaml", "w") as f:
        f.write(f"path: {CALIB}\ntrain: images\nval: images\nnames:\n")
        f.writelines(f"  {i}: {n}\n" for i, n in enumerate(COCO))
    DATA = "/content/calib.yaml"
else:
    DATA = "coco128.yaml"
TEST_FRAMES = sorted(glob.glob(f"{CALIB}/images/*"))[::max(1, len(glob.glob(f'{CALIB}/images/*')) // 20)][:20]
print("calibration data:", DATA, f"({len(os.listdir(CALIB + '/images'))} frames)")

In [ ]:
# 3. Export. Produces several .tflite files in yolov8n_saved_model/ (float32, float16,
#    dynamic-range, int8 with float I/O, full-integer int8 I/O).
from ultralytics import YOLO
import onnx2tf
# onnx2tf 2.x defaults to its new "flatbuffer_direct" writer, whose full-integer model is
# invalid for YOLOv8 (CONCATENATION scale mismatch). Use the classic TensorFlow converter,
# the path Ultralytics' TFLite export was built and tested on.
if not hasattr(onnx2tf, "_orig_convert"):
    onnx2tf._orig_convert = onnx2tf.convert
def _convert_tf_backend(*a, **k):
    k.setdefault("tflite_backend", "tf_converter")
    return onnx2tf._orig_convert(*a, **k)
onnx2tf.convert = _convert_tf_backend
os.chdir("/content")
shutil.rmtree("yolov8n_saved_model", ignore_errors=True)
out = YOLO("yolov8n.pt").export(format="tflite", int8=True, imgsz=IMGSZ, data=DATA, fraction=1.0, opset=17)
print("export returned:", out)
for f in sorted(glob.glob("yolov8n_saved_model/*.tflite")):
    print(f"  {os.path.getsize(f)/1e6:5.1f} MB  {f}")
if not TEST_FRAMES:   # COCO128 fallback: test on its images
    TEST_FRAMES = sorted(glob.glob("/content/datasets/coco128/images/train2017/*.jpg"))[:20]

In [ ]:
# 4. Check every model with the same decode as edge/local_detector.py and compare with float32.
import numpy as np, cv2
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter

def letterbox(img, size):
    h, w = img.shape[:2]; g = min(size / h, size / w)
    nw, nh = int(round(w * g)), int(round(h * g))
    out = np.full((size, size, 3), 114, np.uint8); t, l = (size - nh) // 2, (size - nw) // 2
    out[t:t + nh, l:l + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return out, g, (l, t)

class TFLiteDet:
    def __init__(self, path, conf=0.25, iou=0.45):
        self.m = Interpreter(model_path=path, num_threads=2); self.m.allocate_tensors()
        self.i, self.o = self.m.get_input_details()[0], self.m.get_output_details()[0]
        self.conf, self.iou = conf, iou
        self.nchw = self.i["shape"][1] == 3; self.size = int(self.i["shape"][2 if self.nchw else 1])
    def raw(self, img):
        lb, g, p = letterbox(img, self.size)
        x = cv2.cvtColor(lb, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        if self.nchw: x = x.transpose(2, 0, 1)
        x = x[None]; dt = self.i["dtype"]
        if dt in (np.int8, np.uint8):
            s, z = self.i["quantization"]; x = np.clip(np.round(x / s + z), np.iinfo(dt).min, np.iinfo(dt).max).astype(dt)
        self.m.set_tensor(self.i["index"], x.astype(dt)); self.m.invoke()
        y = self.m.get_tensor(self.o["index"])
        if self.o["dtype"] in (np.int8, np.uint8):
            s, z = self.o["quantization"]; y = (y.astype(np.float32) - z) * s
        y = y[0]
        if y.shape[0] == 84 and y.shape[1] != 84: y = y.T
        return y, g, p
    def detect(self, img):
        y, g, p = self.raw(img)
        b, sc = y[:, :4].copy(), y[:, 4:]
        if b.max() <= 2.0: b *= self.size
        c = sc.argmax(1); cf = sc[np.arange(len(c)), c]; k = cf >= self.conf
        b, cf, c = b[k], cf[k], c[k]
        if not len(cf): return []
        b[:, 0] -= b[:, 2] / 2; b[:, 1] -= b[:, 3] / 2
        sh = b.copy(); sh[:, :2] += c[:, None] * 4096.0
        idx = np.array(cv2.dnn.NMSBoxes(sh.tolist(), cf.tolist(), self.conf, self.iou)).reshape(-1)
        H, W = img.shape[:2]; out = []
        for j in idx:
            x, yy, w, h = b[j]
            out.append((int(c[j]), float(cf[j]), [np.clip((x - p[0]) / g, 0, W), np.clip((yy - p[1]) / g, 0, H),
                                                   np.clip((x + w - p[0]) / g, 0, W), np.clip((yy + h - p[1]) / g, 0, H)]))
        return out

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - ix*iy
    return ix * iy / u if u > 0 else 0

def agreement(test, ref):
    used, m = set(), 0
    for c, s, bx in sorted(test, key=lambda d: -d[1]):
        best, bj = 0, -1
        for j, (c2, s2, b2) in enumerate(ref):
            if j not in used and c2 == c and iou(bx, b2) > best: best, bj = iou(bx, b2), j
        if best >= 0.5: used.add(bj); m += 1
    return m

imgs = [cv2.imread(f) for f in TEST_FRAMES]
ref_path = glob.glob("yolov8n_saved_model/*float32.tflite")[0]
ref = TFLiteDet(ref_path); ref_dets = [ref.detect(im) for im in imgs]
n_ref = sum(map(len, ref_dets))
print(f"reference {os.path.basename(ref_path)}: {n_ref} detections on {len(imgs)} frames\n")

results = {}
for path in sorted(glob.glob("yolov8n_saved_model/*.tflite")):
    if path == ref_path: continue
    d = TFLiteDet(path)
    y, _, _ = d.raw(imgs[0])
    dets = [d.detect(im) for im in imgs]
    n = sum(map(len, dets)); m = sum(agreement(a, b) for a, b in zip(dets, ref_dets))
    box_ok = np.ptp(y[:, :4]) > 0.05            # boxes not collapsed to a constant
    score_ok = y[:, 4:].max() > 0.05            # scores not all rounded to zero
    ok = box_ok and score_ok and n_ref and m / n_ref >= 0.6
    results[path] = (ok, m / max(n_ref, 1), n)
    print(f"{'OK  ' if ok else 'FAIL'} {os.path.basename(path):40s} in {d.i['dtype'].__name__:7s} out {d.o['dtype'].__name__:7s} "
          f"out-quant {d.o['quantization']}  dets {n:3d}  agree {m}/{n_ref} ({m/max(n_ref,1):.0%})"
          + ("" if box_ok else "  <- boxes collapsed") + ("" if score_ok else "  <- scores all zero"))

In [ ]:
# 5. Pick the best fully INT8 model and download it.
#    *_full_integer_quant.tflite : INT8 weights + activations, INT8 input/output  (preferred)
#    *_integer_quant.tflite      : INT8 weights + activations, float32 input/output
#    (*_int8.tflite in Ultralytics 8.3 is only *dynamic-range* quantised: INT8 weights,
#     float activations - not used here.)
order = ["full_integer_quant", "integer_quant"]
best = None
for key in order:
    for path, (ok, agree, n) in results.items():
        if key in os.path.basename(path) and ok and "16" not in os.path.basename(path):
            best = path; break
    if best: break
if not best:
    raise SystemExit("No INT8 model passed the check - send the table above to Claude.")
shutil.copy(best, f"yolov8n_int8_{IMGSZ}.tflite")
print(f"selected {best} (agreement with float32: {results[best][1]:.0%}) -> yolov8n_int8_{IMGSZ}.tflite")
files.download(f"yolov8n_int8_{IMGSZ}.tflite")

### Next
Put `yolov8n_int8_320.tflite` into `models/` of the project and run on the Pi:
```
./scripts/deploy_edge.sh local data/VisDrone2019-VID-val/sequences/uav0000086_00000_v \
    --model models/yolov8n_int8_320.tflite --realtime
```
`edge/local_detector.py` reads INT8 or float input/output and 0..1 or pixel boxes automatically.